Data Reading

In [162]:
import pandas as pd
import json

rows = []

with open("D:\\kevin\\Phase2_Revised\\DM2025-Lab2-Exercise_forked\\Phase3_Data\\final_posts.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    post = item["root"]["_source"]["post"]
    rows.append({
        "type": item["root"]["_type"],
        "post_id": post["post_id"],
        "text": post["text"],
        "hashtags": post["hashtags"]
    })

df = pd.DataFrame(rows)
print(df.head())

   type   post_id                                               text hashtags
0  post  0x61fc95  We got the ranch, loaded our guns and sat up t...       []
1  post  0x35663e  I bet there is an army of married couples who ...       []
2  post  0xc78afe                         This could only end badly.       []
3  post  0x90089c  My sister squeezed a lime in her milk when she...       []
4  post  0xaba820         and that got my head bobbing a little bit.       []


Now that we have loaded the data in, let's split the data into test and training split using the data_identification.csv

In [163]:
data_split = pd.read_csv("D:\\kevin\\Phase2_Revised\\DM2025-Lab2-Exercise_forked\\Phase3_Data\\data_identification.csv")
data_split.head()

,id,split
0,0x61fc95,test
1,0x35663e,train
2,0xc78afe,train
3,0x90089c,train
4,0xaba820,test


In [164]:
train_data_split = data_split[data_split['split'] == 'train']
test_data_split = data_split[data_split['split'] == 'test']

train_df = df[df['post_id'].isin(train_data_split['id'].values)]
test_df = df[df['post_id'].isin(test_data_split['id'].values)]

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 47890
Test size: 16281


Let's add the Associated emotions to the  the train data using emotion.csv

In [165]:
train_target_emotions = pd.read_csv(r"D:\kevin\Phase2_Revised\DM2025-Lab2-Exercise_forked\Phase3_Data\emotion.csv")
train_target_emotions

,id,emotion
0,0x35663e,joy
1,0xc78afe,fear
2,0x90089c,joy
3,0x2ffb63,joy
4,0x989146,joy
...,...,...
47885,0xd740f2,joy
47886,0x99267e,anger
47887,0x4afbe1,anger
47888,0xf5ba78,joy


In [166]:
#Adding the emotion to the corresponding dataframe
train_df = train_df.merge(train_target_emotions, left_on='post_id', right_on='id').drop(columns=['id'])

train_df

,type,post_id,text,hashtags,emotion
0,post,0x35663e,I bet there is an army of married couples who ...,[],joy
1,post,0xc78afe,This could only end badly.,[],fear
2,post,0x90089c,My sister squeezed a lime in her milk when she...,[],joy
3,post,0x2ffb63,Thank you so much❤️,[],joy
4,post,0x989146,Stinks because ive been in this program for a ...,[],joy
...,...,...,...,...,...
47885,post,0xd740f2,why is everybody seem sp serious?,[],joy
47886,post,0x99267e,"You can cross fuck off, its 10f all winter in ...",[],anger
47887,post,0x4afbe1,Guilty Gear actually did that before with Guil...,[],anger
47888,post,0xf5ba78,One of my favorite episodes.,[],joy


In [167]:
#Converting emotion labels to numerical values
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df['emotion_encoded'] = label_encoder.fit_transform(train_df['emotion'])


train_df.head()

,type,post_id,text,hashtags,emotion,emotion_encoded
0,post,0x35663e,I bet there is an army of married couples who ...,[],joy,3
1,post,0xc78afe,This could only end badly.,[],fear,2
2,post,0x90089c,My sister squeezed a lime in her milk when she...,[],joy,3
3,post,0x2ffb63,Thank you so much❤️,[],joy,3
4,post,0x989146,Stinks because ive been in this program for a ...,[],joy,3


Data Cleaning Process

In [168]:
print(train_df.isnull().sum()) #Checking for null values

type               0
post_id            0
text               0
hashtags           0
emotion            0
emotion_encoded    0
dtype: int64


In [169]:
print(train_df.duplicated(subset='post_id').sum()) #Checking for duplicate post_ids

0


In [170]:
# Text length analysis and filtering
print("Text length statistics:")
train_df['text_length'] = train_df['text'].str.len()
print(train_df['text_length'].describe())

# Remove posts that are too short or too long
min_length = 30
max_length = 1000
train_df_cleaned = train_df[(train_df['text_length'] >= min_length) & (train_df['text_length'] <= max_length)]
print(f"Removed {len(train_df) - len(train_df_cleaned)} posts due to length constraints")

#Remaining number of posts
print("Remaining posts after length filtering:", len(train_df_cleaned))

train_df = train_df_cleaned


Text length statistics:
count    47890.000000
mean        75.240405
std         41.664515
min          2.000000
25%         43.000000
50%         72.000000
75%        104.000000
max        703.000000
Name: text_length, dtype: float64
Removed 6810 posts due to length constraints
Remaining posts after length filtering: 41080


In [202]:
# Fix the clean_text function - add return statement
import re

def clean_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()

    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r"[^a-z0-9\s!?']", " ", text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text  # This was missing!

# Re-apply the cleaning
train_df['text_cleaned'] = train_df['text'].apply(clean_text)
test_df['text_cleaned'] = test_df['text'].apply(clean_text)

C:\Users\kevin\AppData\Local\Temp\ipykernel_27312\3307482681.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['text_cleaned'] = test_df['text'].apply(clean_text)


In [203]:
#Checking class imabalance

#Seeing the emotion distribution in the training set
print("Emotion distribution:")
print(train_df['emotion'].value_counts())
print("\nEmotion percentages:")
print(train_df['emotion'].value_counts(normalize=True) * 100)

#Result shows that it's very imbalanace and highly biased to the joy emotion.


Emotion distribution:
emotion
joy         19814
anger        9518
surprise     5491
sadness      3458
fear         1756
disgust      1043
Name: count, dtype: int64

Emotion percentages:
emotion
joy         48.232717
anger       23.169426
surprise    13.366602
sadness      8.417722
fear         4.274586
disgust      2.538948
Name: proportion, dtype: float64


In [177]:
train_df

,type,post_id,text,hashtags,emotion,emotion_encoded,text_length,text_cleaned
0,post,0x35663e,I bet there is an army of married couples who ...,[],joy,3,71,i bet there is an army of married couples who ...
2,post,0x90089c,My sister squeezed a lime in her milk when she...,[],joy,3,127,my sister squeezed a lime in her milk when she...
4,post,0x989146,Stinks because ive been in this program for a ...,[],joy,3,93,stinks because ive been in this program for a ...
5,post,0x111ebf,"The overall response is try and empower women,...",[],anger,0,158,the overall response is try and empower women ...
7,post,0xcc5e81,here’s hoping the same is true for me!,[],joy,3,38,here s hoping the same is true for me!
...,...,...,...,...,...,...,...,...
47884,post,0x34bd3a,"Pet-sitting is over, didn't know I was lonely ...",[],sadness,4,73,pet sitting is over didn't know i was lonely w...
47885,post,0xd740f2,why is everybody seem sp serious?,[],joy,3,33,why is everybody seem sp serious?
47886,post,0x99267e,"You can cross fuck off, its 10f all winter in ...",[],anger,0,56,you can cross fuck off its 10f all winter in w...
47887,post,0x4afbe1,Guilty Gear actually did that before with Guil...,[],anger,0,113,guilty gear actually did that before with guil...


TF-IDF Vectorizer and Prediction

In [190]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion


tfidf = TfidfVectorizer(
    ngram_range=(1, 3),       
    min_df=2,
    max_features=80000,
    sublinear_tf=True,
    strip_accents='unicode'
)

char_tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    min_df=2,
    max_features=50000
)

features = FeatureUnion([
    ("word", tfidf),
    ("char", char_tfidf)
])

X_train = features.fit_transform(train_df['text_cleaned'])
y_train = train_df['emotion_encoded']

In [191]:
from sklearn.model_selection import train_test_split

#Splitting the original training data into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train
)

In [192]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

param_grid = {
    'C': [0.01, 0.1, 1, 5, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [1000, 3000]
}

lr = LogisticRegression(
    class_weight='balanced',
    random_state=42
)

grid = GridSearchCV(
    lr,
    param_grid,
    scoring='f1_macro', 
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train_split, y_train_split)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

#Use the best model
best_lr = grid.best_estimator_

#Evaluate on validation set
y_pred_lr = best_lr.predict(X_val_split)

print("\nLogistic Regression (Tuned) Accuracy:",
      accuracy_score(y_val_split, y_pred_lr))

print("\nClassification Report:")
print(classification_report(
    y_val_split,
    y_pred_lr,
    target_names=label_encoder.classes_
))


Fitting 5 folds for each of 20 candidates, totalling 100 fits


d:\kevin\Phase2_Revised\DM2025-Lab2-Exercise_forked\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Best Parameters: {'C': 1, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV Score: 0.4565774031016835

Logistic Regression (Tuned) Accuracy: 0.6083252190847127

Classification Report:
              precision    recall  f1-score   support

       anger       0.54      0.56      0.55      1903
     disgust       0.23      0.16      0.19       209
        fear       0.46      0.49      0.47       351
         joy       0.74      0.75      0.75      3963
     sadness       0.36      0.33      0.34       692
    surprise       0.48      0.48      0.48      1098

    accuracy                           0.61      8216
   macro avg       0.47      0.46      0.46      8216
weighted avg       0.60      0.61      0.61      8216



In [193]:
#2. MNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

mnb = MultinomialNB()
mnb.fit(X_train_split, y_train_split)

y_pred_mnb = mnb.predict(X_val_split)
print("Multinomial NB Accuracy:", accuracy_score(y_val_split, y_pred_mnb))

Multinomial NB Accuracy: 0.5370009737098345


In [194]:
# 3. SVM with Grid Search
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

param_grid = {
    'C': [0.01, 0.1, 1, 5, 10],
    'loss': ['hinge', 'squared_hinge']
}

svm = LinearSVC(
    class_weight='balanced',
    random_state=42
)

grid = GridSearchCV(
    svm,
    param_grid,
    scoring='f1_macro',  
    cv=5,
    n_jobs=-1,
    verbose=2
)

# Train
grid.fit(X_train_split, y_train_split)

print("Best SVM Parameters:", grid.best_params_)
print("Best CV Score (Macro-F1):", grid.best_score_)

# Use the best model
best_svm = grid.best_estimator_

# Predict on validation set
y_pred_svm = best_svm.predict(X_val_split)

print("\nTuned SVM Accuracy:", accuracy_score(y_val_split, y_pred_svm))
print("\nClassification Report:")
print(classification_report(
    y_val_split,
    y_pred_svm,
    target_names=label_encoder.classes_
))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best SVM Parameters: {'C': 0.1, 'loss': 'squared_hinge'}
Best CV Score (Macro-F1): 0.45832992378256393

Tuned SVM Accuracy: 0.6060126582278481

Classification Report:
              precision    recall  f1-score   support

       anger       0.55      0.55      0.55      1903
     disgust       0.20      0.19      0.19       209
        fear       0.42      0.54      0.47       351
         joy       0.76      0.74      0.75      3963
     sadness       0.34      0.33      0.34       692
    surprise       0.49      0.50      0.49      1098

    accuracy                           0.61      8216
   macro avg       0.46      0.48      0.47      8216
weighted avg       0.61      0.61      0.61      8216



Pre-processing the testing set as well

In [204]:
#We don't remove posts from the testing set, but we just clean up the URLs
test_df['text_cleaned'] = test_df['text'].apply(clean_text)
print(len(test_df[test_df['text'] != test_df['text_cleaned']]), "posts had URLs removed.")

16052 posts had URLs removed.


C:\Users\kevin\AppData\Local\Temp\ipykernel_27312\1783435142.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['text_cleaned'] = test_df['text'].apply(clean_text)


Make predictions on the testing set

In [198]:
# Prepare test data (when ready for final evaluation)
X_test = features.transform(test_df['text_cleaned'])  # Use transform, not fit_transform!

# Make predictions on test set
test_predictions = best_lr.predict(X_test)
test_df['predicted_emotion'] = label_encoder.inverse_transform(test_predictions)

C:\Users\kevin\AppData\Local\Temp\ipykernel_27312\1882245720.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['predicted_emotion'] = label_encoder.inverse_transform(test_predictions)


In [199]:
test_df

,type,post_id,text,hashtags,text_cleaned,predicted_emotion
0,post,0x61fc95,"We got the ranch, loaded our guns and sat up t...",[],we got the ranch loaded our guns and sat up ti...,surprise
4,post,0xaba820,and that got my head bobbing a little bit.,[],and that got my head bobbing a little bit,fear
5,post,0x66e44d,Same. Glad it's not just out store.,[],same glad it's not just out store,joy
6,post,0xc03cf5,Like always i will wait and see thanks for the...,[],like always i will wait and see thanks for the...,joy
8,post,0x02f65a,"There's a bit of room between ""not loving sub-...",[],there's a bit of room between not loving sub z...,anger
...,...,...,...,...,...,...
64146,post,0x0f273c,We all do it sometimes don't worry.,[],we all do it sometimes don't worry,joy
64150,post,0xfc4c5d,This New Year I visited more relatives than us...,[],this new year i visited more relatives than us...,sadness
64157,post,0xb318a3,R u a dad or did ur dad leave u both have bad ...,[],r u a dad or did ur dad leave u both have bad ...,anger
64168,post,0x8f758e,I got my first raspberry from a crowd surfer f...,[],i got my first raspberry from a crowd surfer f...,joy


In [200]:
new_df = test_df[['post_id', 'predicted_emotion']]
new_df.rename(columns={'predicted_emotion': 'emotion', 'post_id': 'id'}, inplace=True)
new_df.to_csv("D:\\kevin\\Phase2_Revised\\DM2025-Lab2-Exercise_forked\\Phase3_Data\\test_predictions.csv", index=False)

C:\Users\kevin\AppData\Local\Temp\ipykernel_27312\1722550282.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.rename(columns={'predicted_emotion': 'emotion', 'post_id': 'id'}, inplace=True)


Bert